# Pipeline de ingestão ate a camada prata de dados de CyberSecurity

## 1 Importações e organização de diretorio

In [4]:
import os
import pandas as pd
import pathlib

Os CSVs de ingestão devem ser postos em data/origin

In [5]:
# Criar as pastas data e subpastas
os.makedirs("../data/bronze", exist_ok=True)
os.makedirs("../data/origin", exist_ok=True) 
os.makedirs("../data/silver", exist_ok=True)

## 2 Ingestão

In [6]:
# Caminho dos arquivos
ORIGIN_PATH = pathlib.Path("../data/origin")
BRONZE_PATH = pathlib.Path("../data/bronze")

Ingestão do csv de incidentes

In [7]:
df_incidents = pd.read_csv(ORIGIN_PATH / "incidents_master.csv")

In [8]:
# Modificar df_incidents: colocar nomes das colunas em maiúsculo e adicionar novas colunas de data e fonte de ingestão.
df_incidents.columns = df_incidents.columns.str.upper()
df_incidents["INGESTION_DATE"] = pd.to_datetime("today").date()
df_incidents["INGESTION_SOURCE"] = "incidents_master.csv"

In [9]:
df_incidents.to_parquet(BRONZE_PATH / "incidents.parquet", index=False)

Ingestão de market_impact.csv

In [10]:
df_market = pd.read_csv(ORIGIN_PATH / "market_impact.csv")

In [11]:
# Modificar df_incidents: colocar nomes das colunas em maiúsculo e adicionar novas colunas de data e fonte de ingestão.
df_market.columns = df_market.columns.str.upper()
df_market["INGESTION_DATE"] = pd.to_datetime("today").date()
df_market["INGESTION_SOURCE"] = "market_impact.csv"

In [12]:
df_market.to_parquet(BRONZE_PATH / "market_impact.parquet", index=False)

Ingestão de financial_impact.csv

In [13]:
df_financial = pd.read_csv(ORIGIN_PATH / "financial_impact.csv")

In [14]:
# Modificar df_incidents: colocar nomes das colunas em maiúsculo e adicionar novas colunas de data e fonte de ingestão.
df_financial.columns = df_financial.columns.str.upper()
df_financial["INGESTION_DATE"] = pd.to_datetime("today").date()
df_financial["INGESTION_SOURCE"] = "financial_impact.csv"

In [15]:
df_financial.to_parquet(BRONZE_PATH / "financial_impact.parquet", index=False)

## 3 Analise de dados

#### analise de incidentes

In [16]:
df_incidents_bronze = pd.read_parquet(BRONZE_PATH / "incidents.parquet")

In [17]:
df_incidents_bronze.isnull().sum()

INCIDENT_ID                   0
COMPANY_NAME                  0
COMPANY_REVENUE_USD           0
COUNTRY_HQ                    0
INDUSTRY_PRIMARY              0
INDUSTRY_SECONDARY          697
EMPLOYEE_COUNT                0
IS_PUBLIC_COMPANY             0
STOCK_TICKER                438
INCIDENT_DATE                 0
INCIDENT_DATE_ESTIMATED       0
DISCOVERY_DATE                0
DISCLOSURE_DATE               0
ATTACK_VECTOR_PRIMARY         0
ATTACK_VECTOR_SECONDARY     639
ATTACK_CHAIN                275
ATTRIBUTED_GROUP            368
ATTRIBUTION_CONFIDENCE      368
DATA_COMPROMISED_RECORDS    248
DATA_TYPE                   248
SYSTEMS_AFFECTED              0
DOWNTIME_HOURS              430
DATA_SOURCE_PRIMARY           0
DATA_SOURCE_SECONDARY       464
DATA_SOURCE_TYPE              0
CONFIDENCE_TIER               0
QUALITY_SCORE                 0
QUALITY_GRADE                 0
REVIEW_FLAG                 780
NOTES                       636
CREATED_AT                    0
UPDATED_

In [18]:
duplicados_id = df_incidents_bronze["INCIDENT_ID"].duplicated().sum()
print(f"\nIDs de incidente repetidos: {duplicados_id}")


IDs de incidente repetidos: 0


In [19]:
df_incidents_bronze.dtypes

INCIDENT_ID                  object
COMPANY_NAME                 object
COMPANY_REVENUE_USD         float64
COUNTRY_HQ                   object
INDUSTRY_PRIMARY             object
INDUSTRY_SECONDARY           object
EMPLOYEE_COUNT                int64
IS_PUBLIC_COMPANY              bool
STOCK_TICKER                 object
INCIDENT_DATE                object
INCIDENT_DATE_ESTIMATED        bool
DISCOVERY_DATE               object
DISCLOSURE_DATE              object
ATTACK_VECTOR_PRIMARY        object
ATTACK_VECTOR_SECONDARY      object
ATTACK_CHAIN                 object
ATTRIBUTED_GROUP             object
ATTRIBUTION_CONFIDENCE       object
DATA_COMPROMISED_RECORDS    float64
DATA_TYPE                    object
SYSTEMS_AFFECTED             object
DOWNTIME_HOURS              float64
DATA_SOURCE_PRIMARY          object
DATA_SOURCE_SECONDARY        object
DATA_SOURCE_TYPE             object
CONFIDENCE_TIER               int64
QUALITY_SCORE               float64
QUALITY_GRADE               

In [20]:
df_incidents_bronze.describe()

,COMPANY_REVENUE_USD,EMPLOYEE_COUNT,DATA_COMPROMISED_RECORDS,DOWNTIME_HOURS,CONFIDENCE_TIER,QUALITY_SCORE
count,8.500000e+02,8.500000e+02,6.020000e+02,420.000000,850.000000,850.000000
mean,1.031337e+10,5.433418e+04,2.708462e+06,107.208762,2.215294,79.963047
std,2.106158e+10,1.266599e+05,2.978069e+07,184.645681,1.202472,12.209540
min,2.424181e+07,6.700000e+01,1.000000e+03,1.830000,1.000000,50.040000
25%,2.327545e+08,1.087000e+03,1.200500e+04,25.822500,1.000000,71.727500
50%,1.261375e+09,6.219500e+03,5.613300e+04,53.600000,2.000000,80.815000
75%,8.692943e+09,4.359475e+04,2.891170e+05,119.800000,3.000000,90.072500
max,1.488980e+11,1.411332e+06,5.497365e+08,1951.620000,4.000000,99.780000


#### análise de market_impact

In [21]:
df_market_bronze = pd.read_parquet(BRONZE_PATH / "market_impact.parquet")

In [22]:
df_market_bronze.isnull().sum()

INCIDENT_ID                          0
STOCK_TICKER                         0
PRICE_7D_BEFORE                      0
PRICE_DISCLOSURE_DAY                 0
PRICE_1D_AFTER                       0
PRICE_7D_AFTER                       0
PRICE_30D_AFTER                      0
VOLUME_AVG_30D_BASELINE              0
VOLUME_DISCLOSURE_DAY                0
SECTOR_INDEX                         0
SECTOR_RETURN_SAME_PERIOD            0
ABNORMAL_RETURN_1D                   0
ABNORMAL_RETURN_7D                   0
ABNORMAL_RETURN_30D                  0
CAR_NEG1_TO_POS1                     0
CAR_0_TO_7                           0
CAR_0_TO_30                          0
CAR_0_TO_90                          0
T_STATISTIC_1D                       0
P_VALUE_1D                           0
T_STATISTIC_30D                      0
P_VALUE_30D                          0
EARNINGS_ANNOUNCEMENT_WITHIN_7D      0
MARKET_CAP_AT_DISCLOSURE             0
VOLUME_RATIO_DISCLOSURE              0
PRE_INCIDENT_VOLATILITY_3

In [23]:
duplicados_id_market = df_market_bronze["INCIDENT_ID"].duplicated().sum()
print(f"\nIDs de incidente repetidos em market_impact: {duplicados_id_market}")


IDs de incidente repetidos em market_impact: 0


In [24]:
df_market_bronze.dtypes

INCIDENT_ID                         object
STOCK_TICKER                        object
PRICE_7D_BEFORE                    float64
PRICE_DISCLOSURE_DAY               float64
PRICE_1D_AFTER                     float64
PRICE_7D_AFTER                     float64
PRICE_30D_AFTER                    float64
VOLUME_AVG_30D_BASELINE              int64
VOLUME_DISCLOSURE_DAY                int64
SECTOR_INDEX                        object
SECTOR_RETURN_SAME_PERIOD          float64
ABNORMAL_RETURN_1D                 float64
ABNORMAL_RETURN_7D                 float64
ABNORMAL_RETURN_30D                float64
CAR_NEG1_TO_POS1                   float64
CAR_0_TO_7                         float64
CAR_0_TO_30                        float64
CAR_0_TO_90                        float64
T_STATISTIC_1D                     float64
P_VALUE_1D                         float64
T_STATISTIC_30D                    float64
P_VALUE_30D                        float64
EARNINGS_ANNOUNCEMENT_WITHIN_7D       bool
MARKET_CAP_

In [25]:
df_market_bronze.describe()

,PRICE_7D_BEFORE,PRICE_DISCLOSURE_DAY,PRICE_1D_AFTER,PRICE_7D_AFTER,PRICE_30D_AFTER,VOLUME_AVG_30D_BASELINE,VOLUME_DISCLOSURE_DAY,SECTOR_RETURN_SAME_PERIOD,ABNORMAL_RETURN_1D,ABNORMAL_RETURN_7D,...,CAR_0_TO_90,T_STATISTIC_1D,P_VALUE_1D,T_STATISTIC_30D,P_VALUE_30D,MARKET_CAP_AT_DISCLOSURE,VOLUME_RATIO_DISCLOSURE,PRE_INCIDENT_VOLATILITY_30D,POST_INCIDENT_VOLATILITY_30D,DAYS_TO_PRICE_RECOVERY
count,358.000000,358.000000,358.000000,358.000000,358.000000,3.580000e+02,3.580000e+02,358.000000,358.000000,358.000000,...,358.000000,358.000000,358.000000,358.000000,358.000000,3.580000e+02,358.000000,358.000000,358.000000,322.000000
mean,129.015642,124.754832,120.293268,120.945279,122.503631,3.556234e+06,9.897671e+06,0.004747,-0.034183,-0.030746,...,-0.012242,-2.594621,0.662270,-1.020512,0.888135,7.796113e+10,2.746851,0.023778,0.037085,109.866460
std,175.589952,169.632368,163.316301,164.186311,166.319908,7.377916e+06,2.063455e+07,0.014757,0.022088,0.026497,...,0.021135,2.172606,0.401876,1.739969,0.258872,1.356710e+11,0.722342,0.008535,0.014843,103.606842
min,5.080000,5.000000,4.610000,4.790000,4.990000,2.746300e+04,8.053000e+04,-0.019971,-0.098128,-0.106112,...,-0.095579,-15.612200,0.000200,-10.355500,0.000200,9.267550e+07,1.508100,0.010135,0.011879,5.000000
25%,13.257500,12.657500,12.110000,12.122500,12.447500,4.439210e+05,1.188034e+06,-0.008347,-0.049536,-0.046946,...,-0.024614,-3.467100,0.266450,-1.911025,0.986000,3.086725e+09,2.114125,0.016342,0.024839,28.000000
50%,43.515000,42.535000,41.050000,40.850000,41.575000,1.166098e+06,3.107902e+06,0.004854,-0.031195,-0.028201,...,-0.010039,-2.167200,0.916400,-0.879650,1.000000,1.929513e+10,2.753400,0.023608,0.034480,57.000000
75%,155.650000,149.607500,148.205000,147.845000,148.980000,2.963769e+06,7.480774e+06,0.018737,-0.018867,-0.010302,...,0.003258,-1.153600,1.000000,0.046925,1.000000,9.017196e+10,3.351200,0.030760,0.047426,177.000000
max,743.830000,741.520000,731.690000,734.290000,753.190000,6.688953e+07,1.728628e+08,0.029954,0.009984,0.025921,...,0.038986,1.185600,1.000000,4.671000,1.000000,1.059593e+12,3.993300,0.039934,0.077072,363.000000


#### análise de financial_impact

In [26]:
df_financial_bronze = pd.read_parquet(BRONZE_PATH / "financial_impact.parquet")

In [27]:
df_financial_bronze.isnull().sum()

INCIDENT_ID                 0
DIRECT_LOSS_USD             0
DIRECT_LOSS_METHOD          0
RANSOM_DEMANDED_USD       572
RANSOM_PAID_USD           692
RANSOM_SOURCE             692
RECOVERY_COST_USD           0
LEGAL_FEES_USD              0
REGULATORY_FINE_USD       646
INSURANCE_PAYOUT_USD      343
TOTAL_LOSS_USD              0
TOTAL_LOSS_METHOD           0
TOTAL_LOSS_LOWER_BOUND      0
TOTAL_LOSS_UPPER_BOUND      0
INFLATION_ADJUSTED_USD      0
CPI_INDEX_USED              0
NOTES                     530
CREATED_AT                  0
UPDATED_AT                  0
INGESTION_DATE              0
INGESTION_SOURCE            0
dtype: int64

In [28]:
duplicados_id_financial = df_financial_bronze["INCIDENT_ID"].duplicated().sum()
print(f"\nIDs de incidente repetidos em financial_impact: {duplicados_id_financial}")


IDs de incidente repetidos em financial_impact: 0


In [29]:
df_financial_bronze.dtypes

INCIDENT_ID                object
DIRECT_LOSS_USD           float64
DIRECT_LOSS_METHOD         object
RANSOM_DEMANDED_USD       float64
RANSOM_PAID_USD           float64
RANSOM_SOURCE              object
RECOVERY_COST_USD         float64
LEGAL_FEES_USD            float64
REGULATORY_FINE_USD       float64
INSURANCE_PAYOUT_USD      float64
TOTAL_LOSS_USD            float64
TOTAL_LOSS_METHOD          object
TOTAL_LOSS_LOWER_BOUND    float64
TOTAL_LOSS_UPPER_BOUND    float64
INFLATION_ADJUSTED_USD    float64
CPI_INDEX_USED             object
NOTES                      object
CREATED_AT                 object
UPDATED_AT                 object
INGESTION_DATE             object
INGESTION_SOURCE           object
dtype: object

In [30]:
df_financial_bronze.describe()

,DIRECT_LOSS_USD,RANSOM_DEMANDED_USD,RANSOM_PAID_USD,RECOVERY_COST_USD,LEGAL_FEES_USD,REGULATORY_FINE_USD,INSURANCE_PAYOUT_USD,TOTAL_LOSS_USD,TOTAL_LOSS_LOWER_BOUND,TOTAL_LOSS_UPPER_BOUND,INFLATION_ADJUSTED_USD
count,7.780000e+02,2.060000e+02,8.600000e+01,7.780000e+02,7.780000e+02,1.320000e+02,4.350000e+02,7.780000e+02,7.780000e+02,7.780000e+02,7.780000e+02
mean,3.652294e+07,8.017672e+06,4.497464e+06,2.618556e+07,7.309469e+06,2.834303e+06,2.665044e+07,7.099600e+07,5.111886e+07,1.059982e+08,7.501910e+07
std,1.196191e+08,1.580389e+07,9.186802e+06,7.816093e+07,2.390878e+07,7.363815e+06,7.593903e+07,2.151881e+08,1.471999e+08,3.283143e+08,2.233519e+08
min,9.000000e+04,5.000000e+04,1.610098e+04,5.703142e+04,1.276379e+04,5.000000e+04,6.280104e+04,1.737931e+05,1.256943e+05,2.752890e+05,1.737931e+05
25%,2.834239e+06,6.578965e+05,4.247265e+05,2.055169e+06,4.748212e+05,2.771979e+05,2.277000e+06,6.166441e+06,4.519930e+06,8.766029e+06,6.489252e+06
50%,8.525071e+06,1.979266e+06,1.129759e+06,6.129803e+06,1.506969e+06,9.141517e+05,6.316200e+06,1.656491e+07,1.185589e+07,2.534531e+07,1.790865e+07
75%,2.609037e+07,6.303181e+06,3.589926e+06,1.840980e+07,4.756604e+06,2.106179e+06,1.844803e+07,5.259585e+07,3.786537e+07,7.729790e+07,5.589355e+07
max,2.302300e+09,7.500000e+07,5.188418e+07,1.238193e+09,3.828183e+08,5.114644e+07,1.054416e+09,3.451548e+09,2.108922e+09,5.872051e+09,3.451548e+09


## 4 Tratamentos para a camada prata

#### Tratamento de incidentes

In [31]:
PRATA_PATH = pathlib.Path("../data/silver")

In [32]:
# convertendo as colunas de data para o formato datetime
date_cols = ['INCIDENT_DATE', 'DISCOVERY_DATE', 'DISCLOSURE_DATE']
for col in date_cols:
    df_incidents_bronze[col] = pd.to_datetime(df_incidents_bronze[col])

In [33]:
# Tratando strings nulas com Unknown quando fizer sentido;
categorical_cols = [
    "INDUSTRY_SECONDARY",
    "ATTACK_VECTOR_SECONDARY",
    "ATTRIBUTED_GROUP",
    "DATA_TYPE",
    "STOCK_TICKER",
    "REVIEW_FLAG",
    "ATTACK_CHAIN",
    "ATTRIBUTION_CONFIDENCE",
]
df_incidents_bronze[categorical_cols] = df_incidents_bronze[categorical_cols].fillna("Unknown")

In [34]:
# 0 para colunas numéricas onde a ausência de valor pode ser interpretada como 0.
df_incidents_bronze['DATA_COMPROMISED_RECORDS'] = df_incidents_bronze['DATA_COMPROMISED_RECORDS'].fillna(0)
df_incidents_bronze['DOWNTIME_HOURS'] = df_incidents_bronze['DOWNTIME_HOURS'].fillna(0)

In [ ]:
# dropando colunas que não são úteis para a análise ou modelagem, como notas internas ou fontes de dados secundárias.
cols_realmente_ruins = ["NOTES", "DATA_SOURCE_SECONDARY", "CREATED_AT", "UPDATED_AT"]
df_incidents_bronze = df_incidents_bronze.drop(columns=cols_realmente_ruins, errors="ignore")

In [ ]:
# criando a coluna target_severidade, onde 1 indica um incidente de alta severidade (downtime > 0 ou mais registros comprometidos que a mediana) e 0 caso contrário.
mediana_records = df_incidents_bronze["DATA_COMPROMISED_RECORDS"].median()
df_incidents_bronze["target_severidade"] = (
    (df_incidents_bronze["DOWNTIME_HOURS"] > 0)
    | (df_incidents_bronze["DATA_COMPROMISED_RECORDS"] > mediana_records)
).astype(int)

In [44]:
df_incidents_bronze.to_parquet(PRATA_PATH / "incidents_prata.parquet", index=False)

#### Tratamento de market

In [45]:
df_market_bronze.head(5)

,INCIDENT_ID,STOCK_TICKER,PRICE_7D_BEFORE,PRICE_DISCLOSURE_DAY,PRICE_1D_AFTER,PRICE_7D_AFTER,PRICE_30D_AFTER,VOLUME_AVG_30D_BASELINE,VOLUME_DISCLOSURE_DAY,SECTOR_INDEX,...,MARKET_CAP_AT_DISCLOSURE,VOLUME_RATIO_DISCLOSURE,PRE_INCIDENT_VOLATILITY_30D,POST_INCIDENT_VOLATILITY_30D,DAYS_TO_PRICE_RECOVERY,NOTES,CREATED_AT,UPDATED_AT,INGESTION_DATE,INGESTION_SOURCE
0,2023-0115-001,BITW,262.07,251.95,245.11,250.34,246.87,19782288,48767234,S&P 500 Information Technology,...,1.181988e+11,2.4652,0.027705,0.052161,255.0,None,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,2026-04-06,market_impact.csv
1,2021-0315-001,SFM,9.37,9.09,8.73,8.78,8.67,458826,1421143,S&P 500 Consumer Discretionary,...,6.489114e+08,3.0973,0.017116,0.027638,324.0,None,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,2026-04-06,market_impact.csv
2,2021-1204-001,SQI,14.60,13.36,13.09,13.28,13.17,230932,354433,S&P 500 Information Technology,...,4.735164e+09,1.5348,0.038209,0.045756,19.0,None,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,2026-04-06,market_impact.csv
3,2021-0213-001,STUT.DE,5.34,5.43,5.27,5.36,5.37,2521343,6491913,S&P 500 Consumer Discretionary,...,2.984412e+09,2.5748,0.027980,0.030972,24.0,None,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,2026-04-06,market_impact.csv
4,2025-0529-001,BAZA,465.19,441.08,410.45,428.36,429.49,1235984,3250371,S&P 500 Consumer Discretionary,...,2.194296e+11,2.6298,0.025212,0.038232,313.0,None,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z,2026-04-06,market_impact.csv
